# 21 cm × Galaxy Cross-Correlation — Part 2: Field Visualisation
## HERA × Euclid (Lightcone)

This notebook loads the simulation outputs produced by `run_simulation.py`
and visualises the simulated fields:

1. **Halo catalogue** — projected positions and mass distribution
2. **SFR distributions** — halo SFR and stellar-mass scaling relations
3. **Lightcone field slices** — brightness temperature, galaxy overdensity,
   and neutral fraction
4. **Wide-format lightcone** — canonical EoR visualisation slice

**Prerequisites:** run `run_simulation.py` (or `sbatch submit_job.sh`) first
to generate `outputs/lightcone_data.h5`.

**References:**
- Davies, Mesinger & Murray (2025) — [arXiv:2504.17254](https://arxiv.org/abs/2504.17254)
- Gagnon-Hartman, Davies & Mesinger (2025) — [arXiv:2502.20447](https://arxiv.org/abs/2502.20447)


## Imports and setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import warnings

warnings.filterwarnings('ignore')

# ── Matplotlib defaults ──────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      300,
    "font.size":       12,
    "axes.labelsize":  13,
    "legend.fontsize": 10,
    "axes.grid":       False,
})


## Configuration — set path to simulation output

Edit `OUTPUT_FILE` if you saved the HDF5 to a different location.


In [ ]:
# ── Path to simulation output (produced by run_simulation.py) ────────────────
OUTPUT_FILE = "../outputs/lightcone_data.h5"

# ── Load all fields and metadata from HDF5 ───────────────────────────────────
with h5py.File(OUTPUT_FILE, 'r') as f:

    # Simulation fields
    brightness_temp_field = f['brightness_temp_field'][:]
    density_field         = f['density_field'][:]
    neutral_fraction      = f['neutral_fraction'][:]
    galaxy_overdensity    = f['galaxy_overdensity'][:]

    # LOS geometry
    lc_redshifts = f['lc_redshifts'][:]
    lc_dist_Mpc  = f['lc_dist_Mpc'][:]

    # Halo catalogue
    halo_masses    = f['halo_catalog/halo_masses'][:]
    halo_coords    = f['halo_catalog/halo_coords'][:]
    stellar_masses = f['halo_catalog/stellar_masses'][:]
    sfr            = f['halo_catalog/sfr'][:]

    # Scalar metadata
    HII_DIM   = int(f.attrs['HII_DIM'])
    BOX_LEN   = float(f.attrs['BOX_LEN'])
    N_z       = int(f.attrs['N_z'])
    L_los     = float(f.attrs['L_los'])
    cell_size = float(f.attrs['cell_size'])
    z_min     = float(f.attrs['z_min'])
    z_max     = float(f.attrs['z_max'])
    z_obs     = float(f.attrs['z_obs'])
    F_21_MHZ  = float(f.attrs['F_21_MHZ'])

print(f'Loaded: brightness_temp_field {brightness_temp_field.shape}')
print(f'        neutral_fraction       {neutral_fraction.shape}')
print(f'        galaxy_overdensity     {galaxy_overdensity.shape}')
print(f'        N_halos                {halo_masses.shape[0]:,}')
print(f'        z = {z_min} -> {z_max},  z_obs = {z_obs}')


## 1  Halo catalogue visualisation

Projected halo positions, mass distribution, and halos overlaid on the
21 cm brightness temperature field at the mid-plane slice.


In [ ]:
# Skip if no halo catalogue was produced (e.g. synthetic fallback)
if halo_masses.shape[0] == 0:
    print('No halo catalogue available — skipping halo plots.')
else:
    # --------------------------------------------------------------------------
    # Subsample for plotting (avoid overplotting large catalogues)
    # --------------------------------------------------------------------------
    
    max_points = 50_000
    rng = np.random.default_rng(42)
    
    N_halos  = halo_coords.shape[0]
    plot_idx = rng.choice(
        N_halos,
        size=min(max_points, N_halos),
        replace=False,
    )
    
    halo_coords_plot = halo_coords[plot_idx]
    halo_masses_plot = halo_masses[plot_idx]
    
    # Coordinates look like grid coordinates if max ~ HII_DIM
    if halo_coords.max() <= HII_DIM + 1:
        print('Interpreting halo coordinates as grid coordinates.')
        halo_pos_mpc = halo_coords_plot * (BOX_LEN / HII_DIM)
    else:
        print('Interpreting halo coordinates as Mpc coordinates.')
        halo_pos_mpc = halo_coords_plot
    
    x = halo_pos_mpc[:, 0]
    y = halo_pos_mpc[:, 1]
    z = halo_pos_mpc[:, 2]
    m = halo_masses_plot
    
    # --------------------------------------------------------------------------
    # Projected halo positions
    # --------------------------------------------------------------------------
    
    plt.figure(figsize=(7, 6))
    
    plt.scatter(
        x,
        y,
        s=np.clip((np.log10(m) - 7.0)**2, 1, 30),
        c=np.log10(m),
        alpha=0.5,
    )
    
    plt.colorbar(label=r"$\log_{10}(M_{\rm halo}/M_\odot)$")
    plt.xlabel("x [Mpc]")
    plt.ylabel("y [Mpc]")
    plt.title(f"Projected 21cmFAST halo catalogue at z = {z_obs}")
    plt.xlim(0, BOX_LEN)
    plt.ylim(0, BOX_LEN)
    plt.gca().set_aspect('equal')
    plt.tight_layout()
    plt.show()
    
    # --------------------------------------------------------------------------
    # Halo mass distribution
    # --------------------------------------------------------------------------
    
    plt.figure(figsize=(7, 5))
    
    plt.hist(
        np.log10(halo_masses[halo_masses > 0]),
        bins=60,
    )
    
    plt.xlabel(r"$\log_{10}(M_{\rm halo}/M_\odot)$")
    plt.ylabel("Number of halos")
    plt.title("Halo mass distribution")
    plt.tight_layout()
    plt.show()
    
    # --------------------------------------------------------------------------
    # Halos in a thin slice over 21-cm brightness temperature
    # --------------------------------------------------------------------------
    
    slice_index       = HII_DIM // 2
    slice_z_mpc       = slice_index * cell_size
    slice_width_cells = 2
    slice_width_mpc   = slice_width_cells * cell_size
    
    # Convert halo coordinates to Mpc for the full catalogue
    if halo_coords.max() <= HII_DIM + 1:
        z_all_mpc = halo_coords[:, 2] * cell_size
        x_all_mpc = halo_coords[:, 0] * cell_size
        y_all_mpc = halo_coords[:, 1] * cell_size
    else:
        z_all_mpc = halo_coords[:, 2]
        x_all_mpc = halo_coords[:, 0]
        y_all_mpc = halo_coords[:, 1]
    
    in_slice     = np.abs(z_all_mpc - slice_z_mpc) < slice_width_mpc
    slice_indices = np.where(in_slice)[0]
    
    print(f'Halos in plotted slice = {len(slice_indices):,}')
    
    max_slice_points = 30_000
    if len(slice_indices) > max_slice_points:
        slice_plot_idx = rng.choice(
            slice_indices,
            size=max_slice_points,
            replace=False,
        )
    else:
        slice_plot_idx = slice_indices
    
    plt.figure(figsize=(7, 6))
    
    plt.imshow(
        brightness_temp_field[:, :, slice_index].T,
        origin='lower',
        extent=[0, BOX_LEN, 0, BOX_LEN],
        interpolation='nearest',
        alpha=0.85,
    )
    
    plt.colorbar(label=r"$\delta T_b$ [mK]")
    
    plt.scatter(
        x_all_mpc[slice_plot_idx],
        y_all_mpc[slice_plot_idx],
        s=np.clip((np.log10(halo_masses[slice_plot_idx]) - 7.0)**2, 1, 30),
        c='white',
        alpha=0.45,
        edgecolors='none',
    )
    
    plt.xlabel("x [Mpc]")
    plt.ylabel("y [Mpc]")
    plt.title(f"Haloes over 21-cm slice at z = {z_obs}")
    plt.xlim(0, BOX_LEN)
    plt.ylim(0, BOX_LEN)
    plt.gca().set_aspect('equal')
    plt.tight_layout()
    plt.show()

## 2  SFR distributions

Halo SFR histogram and scaling relations (SFR vs halo mass, SFR vs stellar
mass) from the 21cmFASTv4 halo catalogue at $z_{\rm obs}$.


In [ ]:
# Skip if no halo catalogue was produced (e.g. synthetic fallback)
if halo_masses.shape[0] == 0:
    print('No halo catalogue available — skipping SFR plots.')
else:
    # --------------------------------------------------------------------------
    # Helper: safely extract values from 21cmFAST Array objects
    # --------------------------------------------------------------------------
    
    def get_21cmfast_array(x):
        """
        Convert a 21cmFAST Array-like object to a NumPy array.
    
        21cmFAST wrapper arrays often store the actual numerical data in `.value`.
        This helper handles both wrapper arrays and ordinary NumPy arrays.
        """
        if hasattr(x, 'value'):
            return np.asarray(x.value)
        return np.asarray(x)
    
    # The arrays are already plain NumPy arrays loaded from HDF5
    # get_21cmfast_array is a no-op here but kept for API consistency
    halo_masses_plot    = get_21cmfast_array(halo_masses)
    stellar_masses_plot = get_21cmfast_array(stellar_masses)
    sfr_plot            = get_21cmfast_array(sfr)
    
    # --------------------------------------------------------------------------
    # Basic diagnostics
    # --------------------------------------------------------------------------
    
    print('Catalogue diagnostics')
    print('---------------------')
    print(f'N_halos = {len(sfr_plot):,}')
    print(f'SFR shape = {sfr_plot.shape}')
    print(f'SFR finite values = {np.isfinite(sfr_plot).sum():,}')
    
    positive_sfr = sfr_plot[np.isfinite(sfr_plot) & (sfr_plot > 0)]
    
    print(f'N halos with SFR > 0 = {len(positive_sfr):,}')
    if len(positive_sfr) > 0:
        print(f'SFR min/max = {positive_sfr.min():.3e} - {positive_sfr.max():.3e} Msun/yr')
        print(f'SFR median  = {np.median(positive_sfr):.3e} Msun/yr')
    
    # --------------------------------------------------------------------------
    # Subsample for scatter plots if catalogue is huge
    # --------------------------------------------------------------------------
    
    max_points = 100_000
    rng2 = np.random.default_rng(42)
    
    valid = (
        np.isfinite(halo_masses_plot)
        & np.isfinite(stellar_masses_plot)
        & np.isfinite(sfr_plot)
        & (halo_masses_plot    > 0)
        & (stellar_masses_plot > 0)
        & (sfr_plot            > 0)
    )
    
    valid_idx = np.where(valid)[0]
    
    if len(valid_idx) > max_points:
        plot_idx2 = rng2.choice(valid_idx, size=max_points, replace=False)
    else:
        plot_idx2 = valid_idx
    
    # ==========================================================================
    # Plot 1: SFR histogram
    # ==========================================================================
    
    plt.figure(figsize=(7, 5))
    
    plt.hist(
        np.log10(positive_sfr),
        bins=80,
    )
    
    plt.xlabel(r"$\log_{10}(\mathrm{SFR}/M_\odot\,\mathrm{yr}^{-1})$")
    plt.ylabel("Number of halos")
    plt.title("Halo catalogue SFR distribution")
    plt.tight_layout()
    plt.show()
    
    # ==========================================================================
    # Plot 2: SFR vs halo mass
    # ==========================================================================
    
    plt.figure(figsize=(7, 5))
    
    plt.scatter(
        np.log10(halo_masses_plot[plot_idx2]),
        np.log10(sfr_plot[plot_idx2]),
        s=2,
        alpha=0.25,
    )
    
    plt.xlabel(r"$\log_{10}(M_{\rm halo}/M_\odot)$")
    plt.ylabel(r"$\log_{10}(\mathrm{SFR}/M_\odot\,\mathrm{yr}^{-1})$")
    plt.title("SFR as a function of halo mass")
    plt.tight_layout()
    plt.show()
    
    # ==========================================================================
    # Plot 3: SFR vs stellar mass
    # ==========================================================================
    
    plt.figure(figsize=(7, 5))
    
    plt.scatter(
        np.log10(stellar_masses_plot[plot_idx2]),
        np.log10(sfr_plot[plot_idx2]),
        s=2,
        alpha=0.25,
    )
    
    plt.xlabel(r"$\log_{10}(M_\star/M_\odot)$")
    plt.ylabel(r"$\log_{10}(\mathrm{SFR}/M_\odot\,\mathrm{yr}^{-1})$")
    plt.title("SFR as a function of stellar mass")
    plt.tight_layout()
    plt.show()

## 3  Visualise the lightcone fields

Three panels showing the lightcone:
- **Left**: transverse (x–y) slice at the mid-point of the LOS
- **Centre**: LOS (x–z) slice showing redshift evolution of 21 cm emission
- **Right**: neutral fraction along the same LOS slice


In [ ]:
mid_z = N_z    // 2
mid_y = HII_DIM // 2

# Extents for imshow
# Standard convention: LOS (comoving distance) on x-axis, transverse on y-axis
extent_xy  = [0, BOX_LEN, 0, BOX_LEN]                         # transverse slice [Mpc]
extent_los = [lc_dist_Mpc[0], lc_dist_Mpc[-1], 0, BOX_LEN]   # LOS on x, transverse on y [Mpc]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: transverse slice at mid-LOS ───────────────────────────────────
# brightness_temp_field[:, :, mid_z] has shape (HII_DIM, HII_DIM) = (x, y)
# .T → (y, x); imshow(origin='lower'): rows=y (↑), cols=x (→)  ✓
vmax_T = np.percentile(np.abs(brightness_temp_field), 99)
im0 = axes[0].imshow(
    brightness_temp_field[:, :, mid_z].T,
    origin='lower', extent=extent_xy,
    cmap='RdBu_r', vmin=-vmax_T, vmax=vmax_T,
)
fig.colorbar(im0, ax=axes[0], label="mK")
axes[0].set_title(
    rf"$\delta T_b$ — transverse slice  ($z \approx {lc_redshifts[mid_z]:.2f}$)"
)
axes[0].set_xlabel("$x$  [Mpc]")
axes[0].set_ylabel("$y$  [Mpc]")

# ── Panel 2: LOS slice — 21 cm evolution ───────────────────────────────────
# brightness_temp_field[:, mid_y, :] has shape (HII_DIM, N_z) = (x_transverse, z_los)
# WITHOUT .T: imshow(origin='lower') → rows=x (y-axis=transverse), cols=z (x-axis=LOS)
# This gives the standard orientation: LOS on x, transverse on y
im1 = axes[1].imshow(
    brightness_temp_field[:, mid_y, :],        # shape (HII_DIM, N_z), no transpose
    origin='lower', extent=extent_los, aspect='auto',
    cmap='RdBu_r', vmin=-vmax_T, vmax=vmax_T,
)
fig.colorbar(im1, ax=axes[1], label="mK")
axes[1].set_title(r"$\delta T_b$ — LOS slice (redshift evolution)")
axes[1].set_xlabel("Comoving distance  [Mpc]")
axes[1].set_ylabel("Transverse $x$  [Mpc]")

# Secondary x-axis on top: redshift labels (twiny, NOT twinx)
ax_twin = axes[1].twiny()
ax_twin.set_xlim(axes[1].get_xlim())
z_ticks = np.linspace(z_min, z_max, 6)
d_ticks = np.interp(z_ticks, lc_redshifts, lc_dist_Mpc)
ax_twin.set_xticks(d_ticks)
ax_twin.set_xticklabels([f'{z:.2f}' for z in z_ticks])
ax_twin.set_xlabel("Redshift $z$")

# ── Panel 3: neutral fraction along same LOS slice ─────────────────────────
# Same orientation fix: shape (HII_DIM, N_z), no transpose
im2 = axes[2].imshow(
    neutral_fraction[:, mid_y, :],             # shape (HII_DIM, N_z), no transpose
    origin='lower', extent=extent_los, aspect='auto',
    cmap='bone', vmin=0, vmax=1,
)
fig.colorbar(im2, ax=axes[2], label=r"$x_{\rm HI}$")
axes[2].set_title("Neutral fraction — LOS slice")
axes[2].set_xlabel("Comoving distance  [Mpc]")
axes[2].set_ylabel("Transverse $x$  [Mpc]")

mean_xHI = np.mean(neutral_fraction)
plt.suptitle(
    rf"Lightcone $z = {z_min}$–${z_max}$,  "
    rf"$\langle x_{{\rm HI}} \rangle = {mean_xHI:.2f}$",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.show()


## 4  Brightness temperature evolution — lightcone slice

Wide-format slice through the lightcone volume, styled after the canonical
21cm reionisation visualisation (cf. Mesinger & Furlanetto 2007).
The x-axis runs along the line of sight from the observer (left, $z_{\rm min}$)
toward the high-redshift universe (right, $z_{\rm max}$); the y-axis is one
transverse spatial direction. A custom EoR colourmap maps dark tones to
fully-ionised regions ($\delta T_b \approx 0$) and warm tones to neutral gas.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# ── EoR-inspired colormap ─────────────────────────────────────────────────
# dark blue-black (ionised, Tb≈0) → blue → cyan → yellow → orange → near-white
# Mirrors the custom colourmap used in 21cmFAST visualisations
_eor_nodes = [
    (0.00, (0.00, 0.00, 0.15)),
    (0.12, (0.00, 0.10, 0.55)),
    (0.30, (0.00, 0.40, 0.85)),
    (0.50, (0.20, 0.78, 0.82)),
    (0.68, (0.95, 0.90, 0.22)),
    (0.85, (0.95, 0.42, 0.00)),
    (1.00, (0.97, 0.97, 0.97)),
]
eor_cmap = LinearSegmentedColormap.from_list(
    "EoR21", [(v, c) for v, c in _eor_nodes], N=256
)

# ── Lightcone slice ───────────────────────────────────────────────────────
# brightness_temp_field has shape (HII_DIM, HII_DIM, N_z).
# Taking [:, mid_y, :] → (HII_DIM, N_z) = (x_transverse, z_los).
# imshow(origin='lower') maps rows→y-axis, cols→x-axis,
# so transverse is on y and LOS is on x.  This is the standard orientation.
T_slice = brightness_temp_field[:, mid_y, :]   # (HII_DIM, N_z)

# Colour limits: clip at zero (emission-only regime at z~6–8)
T_lo = max(0.0, np.percentile(T_slice, 1))
T_hi = np.percentile(T_slice, 99.5)

fig_lc, ax_lc = plt.subplots(figsize=(16, 3.5))

im_lc = ax_lc.imshow(
    T_slice,
    origin='lower',
    extent=[lc_dist_Mpc[0], lc_dist_Mpc[-1], 0, BOX_LEN],
    aspect='auto',
    cmap=eor_cmap,
    vmin=T_lo,
    vmax=T_hi,
)
cbar = fig_lc.colorbar(im_lc, ax=ax_lc, fraction=0.015, pad=0.01)
cbar.set_label(r"$\delta T_b$  [mK]", fontsize=12)

# ── Bottom axis: comoving distance ────────────────────────────────────────
ax_lc.set_xlabel("Comoving distance  [Mpc]", fontsize=12)
ax_lc.set_ylabel("Transverse  $x$  [Mpc]", fontsize=12)

# ── Top axis: redshift ────────────────────────────────────────────────────
ax_top = ax_lc.twiny()
ax_top.set_xlim(ax_lc.get_xlim())
z_ticks = np.linspace(z_min, z_max, 7)
d_ticks = np.interp(z_ticks, lc_redshifts, lc_dist_Mpc)
ax_top.set_xticks(d_ticks)
ax_top.set_xticklabels([f'{z:.2f}' for z in z_ticks])
ax_top.set_xlabel("Redshift  $z$", fontsize=12)

# Observed frequency at each redshift tick (for reference)
f_ticks_MHz = F_21_MHZ / (1 + z_ticks)

f_lo = F_21_MHZ / (1 + z_max)
f_hi = F_21_MHZ / (1 + z_min)
mean_xHI_val = np.mean(neutral_fraction)
ax_lc.set_title(
    rf"21cm brightness temperature lightcone slice   "
    rf"($z = {z_min}$–${z_max}$,  "
    rf"$f_{{\rm obs}} = {f_lo:.0f}$–${f_hi:.0f}$ MHz,  "
    rf"$\langle x_{{\rm HI}} \rangle = {mean_xHI_val:.2f}$)",
    fontsize=12, pad=14,
)
plt.tight_layout()
plt.show()
